In [3]:
import os
import re
import pandas as pd
from pathlib import Path

#load the single csv
DATA_DIR  = Path(r"A:\Coding\PycharmProjects\cryptoguard")
FILE_PATH = DATA_DIR / "data" / "blockchain" / "raw.csv"

if not FILE_PATH.exists():
    raise FileNotFoundError(f"ERROR: {FILE_PATH} not found. Please check your path.")

df = pd.read_csv(FILE_PATH)
print(f"File loaded: {len(df)} rows found.")

#rename sources: fix typos, formatting artifacts, and update names
if 'source' in df.columns:
    df['source'] = df['source'].replace({
        'synthetic_blockchain': 'claude',
        'chatpgt':              'chatgpt',
        'deepseek}':            'deepseek',
        'deepseek"':            'deepseek',
        'gemini"':              'gemini',
    })

print(f"Initial label distribution: {df['label'].value_counts().to_dict()}")

#validate columns
required_cols = ['text', 'label', 'source']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print(f"ERROR: Missing columns {missing}")
else:
    print(f"All required columns present: {list(df.columns)}")

#format cleaning
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'^["\']+ | ["\']+$', '', text)  # strip leading/trailing quotes
    text = re.sub(r'\s+', ' ', text)               # normalise whitespace
    return text.strip()

df['text'] = df['text'].apply(clean_text)

#normalise label to int
df['label'] = pd.to_numeric(df['label'], errors='coerce')
before = len(df)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
df = df[df['label'].isin([0, 1])]
print(f"\nRemoved {before - len(df)} rows with invalid labels")

#length filter
before = len(df)
df = df[df['text'].str.split().str.len().between(10, 500)]
print(f"Removed {before - len(df)} rows outside 10-500 word range")

#english-only filter
def is_english(text):
    if not isinstance(text, str):
        return False
    non_ascii = sum(1 for c in text if ord(c) > 127)
    return non_ascii <= 5   # allow a handful of unicode chars like emoji

before = len(df)
df = df[df['text'].apply(is_english)]
print(f"Removed {before - len(df)} non-English rows")

#exact duplicate removal
before = len(df)
df = df.drop_duplicates(subset=['text'], keep='first')
print(f"\nRemoved {before - len(df)} exact duplicates")

#near-duplicate detection using normalised text
def normalise_for_dedup(text):
    text = text.lower()
    text = re.sub(r'\d+',        'NUM',  text)   # collapse numbers
    text = re.sub(r'0x[0-9a-f]+', 'ADDR', text)  # collapse hex addresses
    text = re.sub(r'[^a-z\s]',   '',    text)   # strip punctuation
    text = re.sub(r'\s+',        ' ',   text).strip()
    return text

df['_normalised'] = df['text'].apply(normalise_for_dedup)
before = len(df)
df = df.drop_duplicates(subset=['_normalised'], keep='first')
df = df.drop(columns=['_normalised'])
print(f"Removed {before - len(df)} near-duplicates")

#pre-balancing stats
print(f"\nPre-Balancing Stats")
print(f"Dataset size before balancing: {len(df)} rows")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")

#class balancing
min_class_size = df['label'].value_counts().min()

df_class0 = df[df['label'] == 0].sample(n=min_class_size, random_state=42)
df_class1 = df[df['label'] == 1].sample(n=min_class_size, random_state=42)

df = pd.concat([df_class0, df_class1], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

assert 'label' in df.columns
assert len(df) == min_class_size * 2
#final stats
print(f"FINAL BALANCED DATASET: {len(df)} rows")
print(f"\nLabel distribution: {df['label'].value_counts().to_dict()}")
print(f"Source distribution:")
print(df['source'].value_counts())

#save
df_final = df[['text', 'label', 'source']]

SAVE_PATH = DATA_DIR / "data" / "blockchain" / "cleaned.csv"
df_final.to_csv(SAVE_PATH, index=False)

#sample preview
print(f"\nSAMPLE PREVIEW (5 random rows)")
for _, row in df_final.sample(5, random_state=1).iterrows():
    label_str = "PHISHING" if row['label'] == 1 else "LEGITIMATE"
    print(f"\n[{label_str}] [{row['source']}]")
    print(f"{row['text'][:200]}")

File loaded: 15621 rows found.
Initial label distribution: {'0': 8345, '1': 7227, 'label': 48, ' non-recycled': 1}
All required columns present: ['text', 'label', 'source']

Removed 49 rows with invalid labels
Removed 3260 rows outside 10-500 word range
Removed 0 non-English rows

Removed 2325 exact duplicates
Removed 1273 near-duplicates

───────────────────────── Pre-Balancing Stats ─────────────────────────
Dataset size before balancing: 8714 rows
Label distribution: {1: 4738, 0: 3976}

FINAL BALANCED DATASET: 7952 rows

Label distribution: {0: 3976, 1: 3976}
Source distribution:
source
gemini      2216
deepseek    1730
chatgpt     1649
grok        1446
claude       911
Name: count, dtype: int64

Saved to A:\Coding\PycharmProjects\cryptoguard\data\blockchain\cleaned.csv

=== SAMPLE PREVIEW (5 random rows) ===

[LEGITIMATE] [grok]
NFT sale complete on Magic Eden: Proceeds 12 SOL received.

[LEGITIMATE] [deepseek]
Optimism airdrop #4 eligibility snapshot taken. Based on your activity 